<a href="https://colab.research.google.com/github/syhennie/data-quality-process/blob/main/features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Настройка окружения

In [1]:
%pip install numpy gensim pandas scipy sdv matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## Предобучение модели векторизации

In [3]:
from gensim.models.fasttext import FastText
from gensim.test.utils import datapath

VECTOR_SIZE = 64

In [ ]:
# Set file names for train and test data
corpus_file = datapath('lee_background.cor')

model = FastText(vector_size=VECTOR_SIZE)

# build the vocabulary
model.build_vocab(corpus_file=corpus_file)

# train the model
model.train(
    corpus_file=corpus_file, epochs=model.epochs,
    total_examples=model.corpus_count, total_words=model.corpus_total_words,
)

with open("../fasttext_lee_background", "w") as file:
    model.save(file.name)

## Загрузка и предобработка данных

In [4]:
df = pd.read_csv('../test.csv')
df.head()

,id,article,highlights
0,92c514c913c0bdfe25341af9fd72b29db544099b,Ever noticed how plane seats appear to be gett...,Experts question if packed out planes are put...
1,2003841c7dc0e7c5b1a248f9cd536d727f27a45a,A drunk teenage boy had to be rescued by secur...,Drunk teenage boy climbed into lion enclosure ...
2,91b7d2311527f5c2b63a65ca98d21d9c92485149,Dougie Freedman is on the verge of agreeing a ...,Nottingham Forest are close to extending Dougi...
3,caabf9cbdf96eb1410295a673e953d304391bfbb,Liverpool target Neto is also wanted by PSG an...,Fiorentina goalkeeper Neto has been linked wit...
4,3da746a7d9afcaa659088c8366ef6347fe6b53ea,Bruce Jenner will break his silence in a two-h...,"Tell-all interview with the reality TV star, 6..."


In [5]:
df = df.iloc[:, 1:]
df.dropna()
df

,article,highlights
0,Ever noticed how plane seats appear to be gett...,Experts question if packed out planes are put...
1,A drunk teenage boy had to be rescued by secur...,Drunk teenage boy climbed into lion enclosure ...
2,Dougie Freedman is on the verge of agreeing a ...,Nottingham Forest are close to extending Dougi...
3,Liverpool target Neto is also wanted by PSG an...,Fiorentina goalkeeper Neto has been linked wit...
4,Bruce Jenner will break his silence in a two-h...,"Tell-all interview with the reality TV star, 6..."
...,...,...
11485,Our young Earth may have collided with a body ...,Oxford scientists say a Mercury-like body stru...
11486,A man facing trial for helping his former love...,Man accused of helping former lover kill woman...
11487,A dozen or more metal implements are arranged ...,Marianne Power tried the tuning fork facial at...
11488,Brook Lopez dominated twin brother Robin with ...,Brooklyn Nets beat the Portland Trail Blazers ...


## Извлечение свойств из набора данных

In [6]:
from gensim.models import Word2Vec, FastText
from gensim.utils import simple_preprocess

model = FastText.load("../fasttext_lee_background")

MAX_TOKENS_PER_ENTRY = 32

def get_vectorised_entries(entries):
    features = []
    for entry in entries:
        tokens = simple_preprocess(entry, min_len=1)
        vectors = [model.wv[token] for token in tokens]
        features.append(np.mean(vectors, axis=0))
    return np.array(features)

def vectorise_entries(entries):
    features = []
    for entry in entries:
        tokens = simple_preprocess(entry)
        vectors = [model.wv[token] for token in tokens]
        length = min(len(vectors), MAX_TOKENS_PER_ENTRY)
        trimmed_vectors = vectors[:length]
        if length < MAX_TOKENS_PER_ENTRY:
            padding_vectors = [np.zeros(VECTOR_SIZE) for _ in range(MAX_TOKENS_PER_ENTRY - length)]
            trimmed_vectors += padding_vectors
        features.append(np.concatenate(trimmed_vectors))
    return np.array(features)

In [7]:
summary_stats = []
features_data = []

df_sample = df.sample(1000)

def calculate_stats(dataframe):
    summary_stats = []
    features_data = []
    for column in dataframe.columns:
        entries = [x for x in dataframe[column]]
        features = get_vectorised_entries(entries)
        vectorised_entries = vectorise_entries(entries)

        # Features-based metrics
        #
        # mean = np.mean(features, axis=0)
        # standard_deviation = np.std(features, axis=0)
        # median = np.median(features, axis=0)
        # asymmetry = stats.skew(features, axis=0)
        # excess = stats.kurtosis(features, axis=0)

        mean = np.mean(vectorised_entries, axis=0)
        standard_deviation = np.std(vectorised_entries, axis=0)
        median = np.median(vectorised_entries, axis=0)
        asymmetry = stats.skew(vectorised_entries, axis=0)
        excess = stats.kurtosis(vectorised_entries, axis=0)

        summary_stats.append({
            'column': column,
            'overall_mean': np.mean(mean),
            'overall_std': np.mean(standard_deviation),
            'std_of_means': np.std(mean),
            'mean_of_medians': np.mean(median),
            'asymmetry_avg': np.mean(asymmetry),
            'excess_avg': np.mean(excess),
            'n_entries': len(entries),
            'vector_dim': vectorised_entries.shape[1]
        })

        features_data.append({
                'column': column,
                'vectors': features
            })
        
    return summary_stats

summary_df = pd.DataFrame(calculate_stats(df_sample))
stats_df = pd.DataFrame(calculate_stats(df))

stats_df

,column,overall_mean,overall_std,std_of_means,mean_of_medians,asymmetry_avg,excess_avg,n_entries,vector_dim
0,article,-0.034556,0.191570,0.431112,-0.032743,-0.061589,-0.273300,11490,2048
1,highlights,-0.032811,0.198002,0.410376,-0.030477,-0.068840,-0.161357,11490,2048


## Генерация синтетического набора данных

In [8]:
def generate_synthetic_vectors(summary_df, method):
    synthetic_data = []

    for i, row in summary_df.iterrows():
        print(i)
        n_samples = row['n_entries']
        vector_dim = row['vector_dim']

        if method == 'normal':
            mean_vector = np.full(vector_dim, row['overall_mean'])
            cov_matrix = np.eye(vector_dim) * (row['std_of_means'] ** 2)

            synthetic_vectors = stats.multivariate_normal.rvs(
                mean=mean_vector,
                cov=cov_matrix,
                size=n_samples
            )

        elif method == 'statistical_adjustment':
            base_vectors = np.random.normal(
                loc=row['overall_mean'],
                scale=row['overall_std'],
                size=(n_samples, vector_dim)
            )

            synthetic_vectors = adjust_statistics(
                base_vectors,
                target_mean=row['overall_mean'],
                target_std=row['overall_std'],
                target_skew=row['asymmetry_avg'],
                target_kurt=row['excess_avg']
            )

        entries = []
        for vector in synthetic_vectors:
            entry = ""
            for i in range(MAX_TOKENS_PER_ENTRY):
                vectorised_token = vector[i * VECTOR_SIZE:(i + 1) * VECTOR_SIZE]
                most_similar_words = model.wv.similar_by_vector(vectorised_token, topn=5)
                # print(most_similar_words)
                for word, _ in most_similar_words:
                    if str(word).isalnum():
                        entry += f" {word}"
            entries.append(entry.strip())

        synthetic_data.append({
            'column': row['column'],
            'vectors': synthetic_vectors,
            'entries': entries,
            'original_stats': row
        })

    return synthetic_data


def adjust_statistics(vectors, target_mean, target_std, target_skew, target_kurt):
    vectors = vectors - np.mean(vectors, axis=0)    # центрирование
    current_std = np.std(vectors, axis=0)   # маштаб
    vectors = vectors * (target_std / (current_std + 1e-8))
    vectors = vectors + target_mean     # сдвиг

    return vectors

In [9]:
def validate_synthetic_data(synthetic_data):
    validation_results = []

    for data in synthetic_data:
        vectors = data['vectors']
        entries = data['entries']
        vectorised_entries = get_vectorised_entries(entries)
        original_stats = data['original_stats']

        synth_mean = np.mean(vectorised_entries, axis=0)
        synth_std = np.std(vectorised_entries, axis=0)
        synth_skew = stats.skew(vectorised_entries, axis=0)
        synth_kurt = stats.kurtosis(vectorised_entries, axis=0)

        validation_results.append({
            'column': original_stats['column'],
            'original_mean': original_stats['overall_mean'],
            'synthetic_mean': np.mean(synth_mean),
            'mean_error': np.abs(np.mean(synth_mean) - original_stats['overall_mean']),

            'original_std': original_stats['overall_std'],
            'synthetic_std': np.mean(synth_std),
            'std_error': np.abs(np.mean(synth_std) - original_stats['overall_std']),

            'original_skew': original_stats['asymmetry_avg'],
            'synthetic_skew': np.mean(synth_skew),
            'skew_error': np.abs(np.mean(synth_skew) - original_stats['asymmetry_avg']),

            'original_kurt': original_stats['excess_avg'],
            'synthetic_kurt': np.mean(synth_kurt),
            'kurt_error': np.abs(np.mean(synth_kurt) - original_stats['excess_avg']),

            'n_samples': len(vectorised_entries),
            'vector_dim': vectorised_entries.shape[1]
        })

    return pd.DataFrame(validation_results)

In [13]:
methods = ['statistical_adjustment']
results = {}

for method in methods:
    print(f"{method}")
    synthetic_data = generate_synthetic_vectors(stats_df, method=method)
    # print(synthetic_data[0]['entries'])
    validation_df = validate_synthetic_data(synthetic_data)


    results[method] = {
        'data': synthetic_data,
        'validation': validation_df
    }
    # print(validation_df[['column', 'mean_error', 'std_error', 'skew_error', 'kurt_error']])
    print(validation_df[['column', 'original_mean', 'synthetic_mean', 'mean_error', 'original_std', 'synthetic_std', 'std_error', 'original_skew', 'synthetic_skew', 'skew_error', 'original_kurt', 'synthetic_kurt', 'kurt_error']])

statistical_adjustment
0
1
       column  original_mean  synthetic_mean  mean_error  original_std  \
0     article      -0.034556       -0.019673    0.014883      0.191570   
1  highlights      -0.032811       -0.019739    0.013072      0.198002   

   synthetic_std  std_error  original_skew  synthetic_skew  skew_error  \
0       0.026402   0.165168      -0.061589       -0.006551    0.055039   
1       0.026788   0.171215      -0.068840       -0.003062    0.065778   

   original_kurt  synthetic_kurt  kurt_error  
0      -0.273300        0.052763    0.326063  
1      -0.161357        0.024697    0.186053  


In [15]:
synthetic_data[0]["entries"]

['2 W No blaze 40 13 Mt SES blaze 2 24 48 SES 5 2 Mt HIV 13 40 40 As Up Mt 13 80 Mt 26 2 18 Mt 23 W 40 No 5 1 2 Mt 26 Mt HIV Up 15 12 5 Mt Bonn On 5 13 150 23 If Mt Up 23 18 26 Mt 2 run If ANZ 2 No 5 13 2 23 W Mt 13 26 48 As 40 23 80 As 25 2 40 13 26 If 2 SES 5 HIV W 5 Mt 13 Bonn 40 W 26 13 14 5 Mt 2 24 oil Mt 13 UN 40 5 SES 15 5 80 2 W As UN No 13 Mt SES 23 13 5 If 23 80 18 Mt 26 Mt Up SES 23 AFP 23 5 15 1 Mt 15 13 Mt W 40 5 HIV 13 80',
 '5 48 23 No Up 40 25 1 If 5 1 26 SES Mt 5 26 2 AFP 21 Up W SES AFP 150 1 As 24 Mt 2 5 15 13 On SES 5 HIV Mt 13 26 Mt 23 1 On 2 5 13 150 run ANZ 26 Mt 1 Mt 23 HIV W 2 40 2 5 13 W 13 Mt 23 18 40 13 5 Up 23 12 2 13 18 Up 150 2 15 Mt 24 80 40 1 As No 14 5 Mt 26 14 5 If 80 drug W Mt If 40 2 80 On If As W As 25 18 23 5 2 80 26 40 Up 5 blaze ANZ 2 5 W drug Kabul 5 13 80 At If 2 As Mt UN 5 AFP blaze 40 80 5 40 Mt On No Mt 40 SES 48 AFP 5 2 40 UN 18',
 'Mt Up 23 25 150 5 40 W 26 13 5 Mt 2 15 1 Mt 150 AFP 2 Mt SES 15 40 AFP 15 40 AFP Nauru As 2 Mt Up blaze 40 1

Per token encoding (32 tokens per entry, 1000 entries)
| col_id |column | mean_error | std_error | skew_error | kurt_error |
| :-- | :-- | --: | --: | --: | --: |
| 0 | article | 0.009493 | 0.070306 | 0.016213 | 0.224872 |
| 1 | highlights | 0.008788 | 0.074577 | 0.016814 | 0.025978 |

Per token encoding (32 tokens per entry, 10000 entries)
| col_id |column | mean_error | std_error | skew_error | kurt_error |
| :-- | :-- | --: | --: | --: | --: |
| 0 | article | 0.009477 | 0.071560 | 0.018867 | 0.097473 |
| 1 | highlights | 0.008795 | 0.074604 | 0.020299 | 0.057919 |

Per token encoding (128 tokens per entry, 1000 entries)
| col_id |column | mean_error | std_error | skew_error | kurt_error |
| :-- | :-- | --: | --: | --: | --: |
| 0 | article | 0.009570 | 0.080066 | 0.019298 | 0.402275 |
| 1 | highlights | 0.000851 | 0.054784 | 0.189310 | 62.405271 |

Per token encoding (256 tokens per entry, 1000 entries)
| col_id |column | mean_error | std_error | skew_error | kurt_error |
| :-- | :-- | --: | --: | --: | --: |
| 0 | article | 0.009448 | 0.083083 | 0.016279 | 0.337313 |
| 1 | highlights | 0.001706 | 0.030596 | 0.560300 | 333.181639 |

In [ ]:
synthetic_data

[{'column': 'article',
  'vectors': array([[-0.02918556, -0.00241555, -0.01755176, ..., -0.01850015,
          -0.02370537, -0.02502471],
         [-0.02091651, -0.0143409 , -0.02476307, ..., -0.01849512,
          -0.01591163, -0.0140214 ],
         [-0.01054276, -0.01340535, -0.01422865, ..., -0.01673924,
          -0.00102861, -0.02292433],
         ...,
         [-0.0035998 , -0.01470775, -0.01273879, ..., -0.01729498,
          -0.02569834, -0.00997742],
         [-0.00747992, -0.01899989, -0.00887265, ..., -0.0187878 ,
          -0.00836075, -0.01032919],
         [-0.01510822, -0.02989641, -0.01921276, ..., -0.01451408,
          -0.02706147, -0.0189524 ]]),
  'original_stats': column              article
  overall_mean      -0.016539
  overall_std         0.00661
  std_of_means       0.203823
  mean_of_medians   -0.016595
  asymmetry_avg      0.034912
  excess_avg         4.487738
  n_entries             11490
  vector_dim              300
  Name: 0, dtype: object},
 {'column':